# The T1 & CT1 support mask: does it keep the signal, and does it fit the prior study?

The stored h5 `mask` (a thresholded brain proxy) was seen to cut into real signal. The loader can
now build the mask from the data instead (`mask_source: "t1ct1"`): the pixels where **this
session's T1 and CT1 are both nonzero**, and nothing else. With `mask_apply: true` every image the
net is conditioned on is multiplied by it -- FLAIR and T2 of this session, and the prior study's
planes according to `mask_guides`:

| `mask_guides` | the prior study is multiplied by |
|---|---|
| `target` | this session's mask -- one support for everything, only as good as the registration |
| `own` | its own T1 & CT1 support at its own slice |
| `none` | nothing |

Three things to check by eye, one section each:

1. **The masks themselves.** Where do the stored mask and the new one disagree?
2. **What each mask removes** from T1, CT1, FLAIR and T2 of the same session. A mask that only
   removes zeros and air is free; one that removes tissue is not.
3. **The prior study.** Does this session's mask fit it? If the two studies are registered, the
   `target` mask should remove about as little of the prior scan as its `own` mask does.

Nothing here is trained. Every image comes through the same loader the configs use, so what is
shown is exactly what a net would be fed. `draw()` resamples the slices.

## Knobs

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

if os.path.basename(os.getcwd()) == "notebooks":     # safe to re-run: only steps up once
    os.chdir("..")
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import datasets                                       # noqa: F401  (registers the loaders)
from datasets.registry import build_loader
from visualization import subplot_images, plot_hist, subplot_hists
from visualization.image import set_display_orient
set_display_orient("radiological")                    # DISPLAY ONLY

ROOT = "../datasets/NYUMets_h5_val"
IMAGE_KEY = "img_raw"
SCALES = [2800.0, 2500.0, 2390.0, 2370.0]             # FLAIR, T1, CT1, T2
SLICE_RANGE = (50, 90)                                # the middle 40 slices; None = every slice
GUIDE_SLICE = "index"                                 # how the prior study's slice is matched
MASK_EPS = 0.0                                        # "nonzero" means |value| > MASK_EPS (scaled units)

N_SLICES = 128                                        # random val slices pooled for the numbers
SEED = 0
N_SHOW = 4                                            # rows in the image panels
SHOW = None                                           # explicit positions, or None = the first N_SHOW

## Load

Three views of the **same** samples (same slice, same prior study -- the loader is deterministic):
the raw images with the stored mask, and the masked images under each `mask_guides` rule.
`guide_mode="other_study"` keeps only patients with at least two studies in this split.

In [ ]:
BASE = dict(name="nyumets_guided", root=ROOT, x0_idx=2, x1_idx=1, cond_idx=[0, 3],
            image_key=IMAGE_KEY, scales=SCALES, guide_mode="other_study", guide_idx=[1, 2],
            guide_slice=GUIDE_SLICE, n_guides=1, deterministic=True, random_flips=False,
            crop_size=None, center_crop=None, batch_size=1, num_workers=0)
if SLICE_RANGE is not None:
    BASE["slice_range"] = list(SLICE_RANGE)
NEW = dict(mask_source="t1ct1", mask_apply=True, mask_eps=MASK_EPS)


def dataset(**kw):
    return build_loader(dict(BASE, **kw), shuffle=False, drop_last=False).dataset


DS = {"raw": dataset(),                               # unmasked images + the stored h5 mask
      "target": dataset(mask_guides="target", **NEW),
      "own": dataset(mask_guides="own", **NEW)}
assert len({len(d) for d in DS.values()}) == 1


def draw(n=N_SLICES, seed=SEED):
    """Pool n random slices into globals. Session images are RAW; M_H5 / M_NEW are the two masks.
    P* are the prior study's T1 / CT1: raw, under this session's mask (_T), under its own (_O)."""
    global T1, CT1, FLAIR, T2, M_H5, M_NEW, PT1, PCT1, PT1_T, PCT1_T, PT1_O, PCT1_O, M_PRIOR, N, IDX
    IDX = np.random.default_rng(seed).choice(len(DS["raw"]), size=min(n, len(DS["raw"])), replace=False)
    acc = {k: [] for k in ("T1", "CT1", "FLAIR", "T2", "M_H5", "M_NEW", "PT1", "PCT1",
                           "PT1_T", "PCT1_T", "PT1_O", "PCT1_O")}
    for i in IDX.tolist():
        r, t, o = DS["raw"][i], DS["target"][i], DS["own"][i]
        acc["CT1"].append(r["x0"][0]); acc["T1"].append(r["x1"][0])
        acc["FLAIR"].append(r["cond"][0]); acc["T2"].append(r["cond"][1])
        acc["M_H5"].append(r["mask"][0] > 0.5); acc["M_NEW"].append(t["mask"][0] > 0.5)
        for tag, s in (("", r), ("_T", t), ("_O", o)):
            acc["PT1" + tag].append(s["guide"][0, 0]); acc["PCT1" + tag].append(s["guide"][1, 0])
        # the masked view must be the raw view times the new mask, or the three are not paired
        assert torch.equal(t["x0"][0], r["x0"][0] * t["mask"][0]), "the views are not paired"
    g = globals()
    for k, v in acc.items():
        g[k] = torch.stack(v)
    M_PRIOR = (PT1_O != 0) & (PCT1_O != 0)            # the prior study's own T1 & CT1 support
    N = len(IDX)
    print("%d slices pooled (of %d), frame %s" % (N, len(DS["raw"]), tuple(T1.shape[-2:])))


draw()

## Helpers

In [ ]:
def rows():
    return list(range(min(N_SHOW, N))) if SHOW is None else list(SHOW)


def hi(img, m=None, q=99.5):
    """Display ceiling of one contrast: a high percentile inside the mask."""
    v = img[m] if m is not None else img
    v = v[v > 0]
    return float(np.percentile(v.numpy(), q)) if v.numel() else 1.0


def removed_energy(img, m):
    """Share of sum(img^2) that multiplying by mask m throws away, pooled over the slices."""
    tot = float((img ** 2).sum())
    return float((img ** 2 * (~m)).sum()) / max(tot, 1e-30)


def removed_per_slice(img, m):
    tot = (img ** 2).sum((1, 2)).clamp_min(1e-30)
    return ((img ** 2 * (~m)).sum((1, 2)) / tot).numpy()


def table(rows_, cols, fmt="%11.4f", title=None):
    w = max(len(k) for k in rows_) + 2
    if title:
        print(title)
    print(" " * w + "".join("%11s" % c for c in cols))
    for k, r in rows_.items():
        print(k.ljust(w) + "".join(fmt % r[c] for c in cols))


SESSION = {"T1": lambda: T1, "CT1": lambda: CT1, "FLAIR": lambda: FLAIR, "T2": lambda: T2}

## 1. The two masks

The last column is the disagreement: **red** = kept by the new mask but removed by the stored one
(signal the old mask was cutting, if there is tissue there), **blue** = kept by the stored mask but
outside the T1 & CT1 support.

In [ ]:
R = rows()
V_T1, V_CT1 = hi(T1, M_NEW), hi(CT1, M_NEW)
diff = M_NEW.float() - M_H5.float()
subplot_images([[T1[i], CT1[i], M_H5[i].float(), M_NEW[i].float(), diff[i]] for i in R],
               col_titles=["T1", "CT1", "stored h5 mask", "T1 & CT1 support", "new - stored"],
               cmap=["gray", "gray", "gray", "gray", "bwr"],
               vmin=[0.0, 0.0, 0.0, 0.0, -1.0], vmax=[V_T1, V_CT1, 1.0, 1.0, 1.0],
               row_labels=["#%d" % i for i in R], panel_size=(2.6, 2.8),
               suptitle="the stored mask against the T1 & CT1 support")

a, b = M_NEW, M_H5
print("mask covers  stored %.1f%%   new %.1f%%   of the frame" % (100 * b.float().mean(), 100 * a.float().mean()))
print("of the T1 & CT1 support, the stored mask removes %.1f%%" % (100 * float((a & ~b).sum()) / max(float(a.sum()), 1)))
print("of the stored mask, %.2f%% lies outside the T1 & CT1 support" % (100 * float((b & ~a).sum()) / max(float(b.sum()), 1)))
holes = [float((~M_NEW[i] & M_H5[i]).sum()) for i in range(N)]
print("pixels inside the stored mask where T1 or CT1 is exactly zero: median %.0f, max %.0f per slice "
      "(isolated zeros punch holes in the new mask; raise MASK_EPS only if these are noise)"
      % (np.median(holes), max(holes)))

## 2. What each mask removes from this session's images

For each contrast: the image, then the part a mask would **zero out**, on the image's own window.
A removed panel that is black means the mask costs nothing for that contrast; anatomy visible in it
is signal being thrown away.

By construction the new mask removes nothing from T1 where CT1 has signal and vice versa, so its
T1 and CT1 "removed" panels show only where the two supports differ. FLAIR and T2 are the real
test: they are cut to a support they did not define.

In [ ]:
def show_removed(m, name):
    cols, titles, cm, lo, up = [], [], [], [], []
    for c, get in SESSION.items():
        v = hi(get(), M_NEW)
        titles += [c, c + " removed"]
        cm += ["gray", "gray"]; lo += [0.0, 0.0]; up += [v, v]
    subplot_images([[x for c, get in SESSION.items() for x in (get()[i], get()[i] * (~m[i]))] for i in R],
                   col_titles=titles, cmap=cm, vmin=lo, vmax=up,
                   row_labels=["#%d" % i for i in R], panel_size=(2.1, 2.3),
                   suptitle="removed by the " + name)


show_removed(M_H5, "stored h5 mask")
show_removed(M_NEW, "T1 & CT1 support mask")

table({"stored h5 mask": {c: 100 * removed_energy(get(), M_H5) for c, get in SESSION.items()},
       "T1 & CT1 support": {c: 100 * removed_energy(get(), M_NEW) for c, get in SESSION.items()}},
      list(SESSION), fmt="%10.3f%%", title="\nshare of each contrast's energy the mask removes (pooled)")

In [ ]:
# How bright is what gets removed? If the removed pixels sit at the same intensities as the kept
# ones, the mask is cutting tissue; if they pile up near zero, it is cutting air.
panels = {}
for c, get in SESSION.items():
    img = get()
    series = {"kept": img[M_NEW].numpy(), "removed by new": img[~M_NEW & (img != 0)].numpy(),
              "removed by stored": img[~M_H5 & (img != 0)].numpy()}
    panels[c] = {k: v for k, v in series.items() if v.size}
subplot_hists(panels, bins=120, p=(0, 99.5), log=True, ncols=4, xlabel="intensity (scaled units)",
              suptitle="intensity of the nonzero pixels each mask keeps and removes")

## 3. The prior study

The same question for the prior study's T1 and CT1, under the two `mask_guides` rules. The last
column compares this session's mask with the prior study's own support: **red** = inside this
session's mask only, **blue** = inside the prior study's support only. A thin outline is ordinary
between two visits; a thick crescent on one side is a shift, i.e. residual misregistration, and
then the `target` rule is cutting through the prior scan.

In [ ]:
V_P = hi(PCT1, M_PRIOR)
d2 = M_NEW.float() - M_PRIOR.float()
subplot_images([[PCT1[i], PCT1_T[i], PCT1[i] * (~M_NEW[i]), PCT1_O[i], PCT1[i] * (~M_PRIOR[i]), d2[i]]
                for i in R],
               col_titles=["prior CT1", "x this session's mask", "removed (target)",
                           "x its own mask", "removed (own)", "this mask - prior support"],
               cmap=["gray"] * 5 + ["bwr"],
               vmin=[0.0] * 5 + [-1.0], vmax=[V_P] * 5 + [1.0],
               row_labels=["#%d" % i for i in R], panel_size=(2.5, 2.7),
               suptitle="prior study's CT1 under mask_guides = target / own")

PRIOR = {"prior T1": PT1, "prior CT1": PCT1}
table({"target (this session's mask)": {c: 100 * removed_energy(v, M_NEW) for c, v in PRIOR.items()},
       "own (prior T1 & CT1 support)": {c: 100 * removed_energy(v, M_PRIOR) for c, v in PRIOR.items()}},
      list(PRIOR), fmt="%10.3f%%", title="\nshare of the prior scan's energy the mask removes (pooled)")

In [ ]:
# Per slice: how well does this session's mask overlap the prior study's support, and how much of
# the prior CT1 does it cost? A long tail here is a subset of badly aligned studies.
inter = (M_NEW & M_PRIOR).sum((1, 2)).float()
dice = (2 * inter / (M_NEW.sum((1, 2)) + M_PRIOR.sum((1, 2))).clamp_min(1).float()).numpy()
lost_t = 100 * removed_per_slice(PCT1, M_NEW)
lost_o = 100 * removed_per_slice(PCT1, M_PRIOR)
print("Dice(this session's mask, prior support): median %.3f, p10 %.3f, min %.3f"
      % (np.median(dice), np.percentile(dice, 10), dice.min()))
print("prior CT1 energy removed per slice:  target median %.2f%%, p90 %.2f%%, max %.2f%%   |   "
      "own median %.2f%%" % (np.median(lost_t), np.percentile(lost_t, 90), lost_t.max(), np.median(lost_o)))
subplot_hists({"Dice between the two supports": dice,
               "prior CT1 energy removed (%)": {"target": lost_t, "own": lost_o}},
              bins=40, xlabel=["Dice", "% of energy"], suptitle="per slice, %d slices" % N)
worst = np.argsort(dice)[:N_SHOW]
print("lowest-Dice slices (set SHOW to look at them):", worst.tolist())

## Reading it, and using it

* **Section 2 decides the mask.** If the new mask's FLAIR and T2 "removed" panels are black and
  its row of the table is near zero, it is safe to apply to everything.
* **Section 3 decides `mask_guides`.** If `target` removes little more than `own` and the Dice
  histogram is tight near 1, use `target`: one support for every input. If there is a tail, either
  use `own` or fix those studies' registration first.

In a config's `data.train` and `data.val` blocks:

```json
"mask_source": "t1ct1", "mask_apply": true, "mask_guides": "target", "slice_range": [50, 90]
```

These keys belong to the `nyumets_guided` and `nyumets_synth` loaders. A config that uses the plain
`"name": "i2sb"` loader has to switch to `"name": "nyumets_guided"` (with `"guide_mode": "none"`
it returns the identical batch) to use them. `training.use_mask` is separate: it decides whether
the LOSS and the metrics are restricted to the mask, whichever mask the loader returns.

`draw(n, seed)` pools a different set of slices; `SHOW = [..]` picks the rows of the image panels.